In [9]:
import os
import numpy as np
import rasterio
import spectral.io.envi as envi

hdr_file = r"E:\wenqu\2024_data\site7\site7.hdr"
out_tif  = r"E:\wenqu\2024_data\rgb\site7_rgb.tif"

target_wavelengths = [671.11, 540.44, 480.64]

# find ENVI data file
base = os.path.splitext(hdr_file)[0]
candidate_files = [base, base+".img", base+".dat", base+".bin"]

data_file = None
for f in candidate_files:
    if os.path.exists(f):
        data_file = f
        break

if data_file is None:
    raise FileNotFoundError("Cannot find ENVI data file")

# read wavelengths
img = envi.open(hdr_file, data_file)
metadata = img.metadata

wavelengths = np.array([float(w.strip()) for w in metadata["wavelength"]])

# find closest bands
band_indices0 = [int(np.argmin(np.abs(wavelengths - tw))) for tw in target_wavelengths]
band_indices1 = [int(i + 1) for i in band_indices0]

for tw, b0 in zip(target_wavelengths, band_indices0):
    print(f"Target {tw} nm -> band {b0+1}, wavelength = {wavelengths[b0]}")

# read raster and build RGB
with rasterio.open(data_file) as src:

    red   = src.read(int(band_indices1[0])).astype(np.float32)
    green = src.read(int(band_indices1[1])).astype(np.float32)
    blue  = src.read(int(band_indices1[2])).astype(np.float32)

    rgb = np.stack([red, green, blue])

    rgb_out = np.zeros((3, src.height, src.width), dtype=np.uint8)

    for i in range(3):

        band = rgb[i]

        if src.nodata is not None:
            valid = (band != src.nodata) & np.isfinite(band)
        else:
            valid = np.isfinite(band)

        vals = band[valid]

        p2, p98 = np.percentile(vals, (2, 98))

        if p98 <= p2:
            rgb_out[i] = np.zeros_like(band, dtype=np.uint8)
            continue

        rgb_out[i] = (
            np.clip((band - p2) / (p98 - p2), 0, 1) * 255
        ).astype(np.uint8)

    profile = src.profile.copy()
    profile.update(
        driver="GTiff",
        count=3,
        dtype="uint8",
        nodata=None
    )

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(rgb_out)

print("Saved:", out_tif)

Target 671.11 nm -> band 124, wavelength = 671.106
Target 540.44 nm -> band 65, wavelength = 540.439
Target 480.64 nm -> band 38, wavelength = 480.641
Saved: E:\wenqu\2024_data\rgb\site7_rgb.tif
